# V3

In [1]:
# D:\Zenith AI\MAS_UseCase_V2\main_v31.py
from __future__ import annotations
import os, json, re
from pathlib import Path
from typing import Dict, Any, List

from tools._paths import find_data_dir
from schemas.datasets import discover_csv_meta, meta_as_json
from runtime.registry import load_registry, save_registry, set_datasets_metadata, register_tool, register_agent
from planning.agent_planner import plan
from codegen.tool_codegen import generate_tool_module
from codegen.agent_codegen import generate_agent_module

ROOT = Path.cwd()
DATA_DIR = find_data_dir(ROOT)
CREATED_TOOLS_DIR = ROOT / "created_tools"
CREATED_AGENTS_DIR = ROOT / "created_agents"
CREATED_TOOLS_DIR.mkdir(exist_ok=True)
CREATED_AGENTS_DIR.mkdir(exist_ok=True)
(CREATED_TOOLS_DIR / "__init__.py").write_text("", encoding="utf-8")
(CREATED_AGENTS_DIR / "__init__.py").write_text("", encoding="utf-8")

os.environ.setdefault("ZENITH_DEBUG_PROMPTS", "1")  # show prompts by default in notebooks

D:\Zenith AI\MAS_UseCase_V3


'1'

In [2]:

# Existing tools/agents declared here for the planner
EXISTING_TOOLS = [
    {"name": "sales_this_year", "description": "Return sales rows for the most recent year joined with customers to add region.", "inputs": [], "outputs": [{"name": "sales", "type": "pd.DataFrame", "description": "year rows"}], "datasets": ["sales","customers"], "assumptions": ["sales has date, customer_id, quantity, price", "customers has customer_id, region"]},
    {"name": "reviews_this_year","description": "Return review rows for the most recent year joined with customers to add region.","inputs": [],"outputs": [{"name": "reviews", "type": "pd.DataFrame", "description": "year rows"}],"datasets": ["reviews","customers"],"assumptions": ["reviews has review_date, product_id, customer_id, rating"]}
]
EXISTING_AGENTS = [
  {"name": "SalesThisYearAnalyst", "goal": "Summarize units and revenue for detected year; LLM-based markdown analysis.", "uses_tools": ["sales_this_year"], "inputs": [], "outputs": [{"name": "summary", "type": "markdown"}, {"name": "table", "type": "pd.DataFrame"}]},
  {"name": "ReviewThisYearInvestigator", "goal": "Summarize ratings distribution and key review signals for detected year; LLM-based markdown.", "uses_tools": ["reviews_this_year"], "inputs": [], "outputs": [{"name": "summary", "type": "markdown"}, {"name": "table", "type": "pd.DataFrame"}]},
]


def read_business_yaml() -> str:
    path = ROOT / "schemas" / "business_context.yaml"
    return path.read_text(encoding="utf-8") if path.exists() else "brand: NovaMobile"

# ---------- Code sanitizers ----------

def _sanitize_generated_code(code: str) -> str:
    code = re.sub(r"read_csv\(([^\)]*?),\s*errors\s*=\s*[^,\)]+", r"read_csv(\1", code)
    code = re.sub(r"date_parser\s*=\s*[^,\)]+,?\s*", "", code)
    code = code.replace(".use()", "")
    return code

def _remap_imports_for_created_tools(code: str, created_tool_names: set[str]) -> str:
    def snake(name: str) -> str:
        s = re.sub(r"[^0-9a-zA-Z_]+", "_", name.strip())
        s = re.sub(r"_{2,}", "_", s).strip("_")
        return s.lower() or "unnamed"
    for t in sorted(created_tool_names):
        t_file = snake(t)
        code = re.sub(rf"\bfrom\s+tools\.{t_file}\s+import\s+{t}\b", f"from created_tools.{t_file} import {t}", code)
        code = re.sub(rf"\bimport\s+tools\.{t_file}\s+as\s+(\w+)\b", f"import created_tools.{t_file} as \\1", code)
    return code


def write_file(path: Path, code: str) -> None:
    path.write_text(code, encoding="utf-8")
    print("Wrote:", path)


In [3]:
# ---------- Generation orchestrator ----------

def ensure_generated(plan_dict: Dict[str, Any], datasets_meta_json: str) -> None:
    reg = load_registry()

    # Persist the latest datasets metadata for all components
    meta = discover_csv_meta(DATA_DIR)
    set_datasets_metadata(meta)

    # 1) Tools
    existing_tool_names = {t["name"] for t in EXISTING_TOOLS}
    for spec in plan_dict.get("tools_to_create", []):
        name = spec.get("name")
        if not name:
            continue
        if name in reg.get("tools", {}):
            print(f"[skip] tool already registered: {name}")
            continue
        mod = generate_tool_module(spec, datasets_meta_json, ", ".join(sorted(existing_tool_names)))
        filename = mod.get("filename") or f"{name}.py"
        path = CREATED_TOOLS_DIR / filename
        code = _sanitize_generated_code(mod.get("code", ""))
        write_file(path, code)
        register_tool(name, str(path), description=spec.get("description"))

    # 2) Agents
    reg = load_registry()  # refresh to see newly registered tools
    created_tool_names = set(reg.get("tools", {}).keys()) - {t["name"] for t in EXISTING_TOOLS}
    all_tool_names = sorted({t["name"] for t in EXISTING_TOOLS}.union(created_tool_names))

    for spec in plan_dict.get("agents_to_create", []):
        name = spec.get("name")
        if not name:
            continue
        if name in reg.get("agents", {}):
            print(f"[skip] agent already registered: {name}")
            continue
        # Filter uses_tools to valid tool names only (avoid agent names here)
        valid_tool_names = set(all_tool_names)
        spec["uses_tools"] = [t for t in spec.get("uses_tools", []) if t in valid_tool_names]

        mod = generate_agent_module(
            spec,
            datasets_meta_json,
            ", ".join(all_tool_names),
            ", ".join(sorted(created_tool_names)),
        )
        filename = mod.get("filename") or f"{name}.py"
        path = CREATED_AGENTS_DIR / filename
        code = _sanitize_generated_code(mod.get("code", ""))
        code = _remap_imports_for_created_tools(code, created_tool_names)
        write_file(path, code)
        register_agent(name, str(path), description=spec.get("goal"))



In [4]:
# ---------- MAIN ----------
if __name__ == "__main__":
    objective = os.environ.get(
        "ZENITH_OBJECTIVE",
        "increse sales profit"
    )
    memory = {"region_focus": "West", "stakeholder": "Retail Ops", "format": "bullet_md"}

    meta = discover_csv_meta(DATA_DIR)
    datasets_meta_json = meta_as_json(meta)
    business_yaml = read_business_yaml()

    print("\n[INFO] Objective:", objective)
    print("[INFO] Memory:", memory)
    print("[INFO] Discovered datasets:", list(meta.keys()))

    plan_dict = plan(
        objective=objective,
        datasets_meta_json=datasets_meta_json,
        business_yaml=business_yaml,
        existing_tools_json=json.dumps(EXISTING_TOOLS, indent=2),
        existing_agents_json=json.dumps(EXISTING_AGENTS, indent=2),
    )

    print("\n=== PLAN ===\n", json.dumps(plan_dict, indent=2))
    ensure_generated(plan_dict, datasets_meta_json)

    # ---- SKIP EXECUTION ----
    # If you only want creation of tools/agents, stop here.
    # To run the generated runbook, uncomment below:
    #
    # from runtime.runbook import execute_runbook
    # results = execute_runbook(plan_dict.get("runbook", []), objective, memory)
    #
    # for item in results:
    #     print("\n=== ", item.get("agent"), " ===")
    #     print(item.get("summary"))
    #     print("\nFacts:", json.dumps(item.get("facts", {}), indent=2))
    #     tbl = item.get("table")
    #     try:
    #         import pandas as pd
    #         if isinstance(tbl, pd.DataFrame):
    #             print("\nTable head:\n", tbl.head())
    #     except Exception:
    #         pass


[INFO] Objective: increse sales profit
[INFO] Memory: {'region_focus': 'West', 'stakeholder': 'Retail Ops', 'format': 'bullet_md'}
[INFO] Discovered datasets: ['customers', 'kpi_results', 'reviews', 'sales']


==================== DEBUG: PLANNER PROMPT ====================

You are the **Agent Planner** for NovaMobile Retail.

Return ONLY valid JSON (no fences) that matches this schema:
{
  "goals": ["..."],
  "tools_to_create": [
    {
      "name": "...",
      "description": "Business purpose + exact logic (filters, windows, joins, dedupe), data-quality checks, validations, failure modes, performance notes.",
      "inputs": [],
      "outputs": [{"name":"...","type":"pd.DataFrame","description":"...","schema":"optional"}],
      "datasets": ["..."],
      "assumptions": ["List all hard assumptions explicitly"]
    }
  ],
  "agents_to_create": [
    {
      "name":"...",
      "goal":"Business role & how it moves KPIs using tools",
      "uses_tools":["..."],
      "inputs": [],
  

In [22]:
# ---------- MAIN ----------
objective = os.environ.get(
    "ZENITH_OBJECTIVE",
    "Increase this year's revenue in West by 8% while keeping stockouts < 2%"
)
memory = {"region_focus": "West", "stakeholder": "Retail Ops", "format": "bullet_md"}

meta = discover_csv_meta(DATA_DIR)
datasets_meta_json = meta_as_json(meta)
business_yaml = read_business_yaml()

print("\n[INFO] Objective:", objective)
print("[INFO] Memory:", memory)
print("[INFO] Discovered datasets:", list(meta.keys()))

plan_dict = plan(
    objective=objective,
    datasets_meta_json=datasets_meta_json,
    business_yaml=business_yaml,
    existing_tools_json=json.dumps(EXISTING_TOOLS, indent=2),
    existing_agents_json=json.dumps(EXISTING_AGENTS, indent=2),
)


[INFO] Objective: Increase this year's revenue in West by 8% while keeping stockouts < 2%
[INFO] Memory: {'region_focus': 'West', 'stakeholder': 'Retail Ops', 'format': 'bullet_md'}
[INFO] Discovered datasets: ['customers', 'kpi_results', 'reviews', 'sales']
